# MR DRE — Kaggle baseline (chọn câu theo ID)

Sử dụng mã nguồn repo đã cập nhật. Chỉnh `QUESTION_IDS` ở ô cấu hình để chọn câu hỏi trong `ResearcherBench.jsonl`. Mỗi câu chạy đủ 19 bước trước khi chuyển câu tiếp theo. Kết quả tự đặt tên `baseline-results-<các ID>.zip`.

Trên Kaggle: bật Internet, chọn GPU T4 × 2; thêm dataset chứa ZIP repo đã cập nhật vào Add Input. Có thể đính kèm ZIP kết quả của lần chạy trước để khôi phục checkpoint khi các ID và cấu hình khớp.


In [ ]:
REPO_DIR = "/kaggle/working/ReviFence-MRDRE"
OUTPUT_ROOT = "/kaggle/working/output"
SOURCE_ZIP = ""     # Đường dẫn ZIP repo trong Kaggle Input; để trống để tự tìm.
RESUME_ZIP = ""     # Đường dẫn ZIP checkpoint; để trống để tự tìm checkpoint cùng câu.
QUESTION_IDS = ("1",) # Ví dụ ("8", "9", "10", "11", "12", "13")
MODEL = "qwen3:4b-instruct"
ODR_REF = "1b7d2e80db9faa586165c60e09096dbbfd483a64"
FEEDBACK = "all"
EXTRA_EVAL = "all"
BUDGET_MINUTES = 450
TASK_RETRIES = 2

QUESTION_IDS = tuple(str(x) for x in QUESTION_IDS)
assert QUESTION_IDS and len(QUESTION_IDS) == len(set(QUESTION_IDS)), "QUESTION_IDS phải khác rỗng, không trùng"
COUNT = len(QUESTION_IDS)
CHECKPOINT_ZIP = "/kaggle/working/baseline-results-" + "-".join(QUESTION_IDS) + ".zip"


In [ ]:
import os, sys, json, shutil, zipfile, subprocess, hashlib
from pathlib import Path

input_dir = Path('/kaggle/input')
repo = Path(REPO_DIR)
source_dir = None
source_zip = Path(SOURCE_ZIP) if SOURCE_ZIP else None

def source_archive(path):
    try:
        with zipfile.ZipFile(path) as archive:
            roots = sorted(set(name.removesuffix('pyproject.toml') for name in archive.namelist()
                               if name.endswith('/pyproject.toml')))
            return len(roots) == 1 and any(name == roots[0] + 'src/mrdre_baseline/runtime/runner.py'
                                           for name in archive.namelist())
    except (OSError, zipfile.BadZipFile):
        return False

if source_zip is None:
    extracted = sorted({p.parent for p in input_dir.rglob('pyproject.toml')
                        if (p.parent / 'src/mrdre_baseline/runtime/runner.py').is_file()})
    if len(extracted) == 1:
        source_dir = extracted[0]
    elif len(extracted) > 1:
        raise RuntimeError(f'Nhiều repo trong Kaggle Input: {extracted}; hãy giữ một bản')
    else:
        archives = [p for p in input_dir.rglob('*.zip') if source_archive(p)]
        if len(archives) != 1:
            raise RuntimeError(f'Cần đúng một ZIP mã nguồn repo trong Kaggle Input; tìm thấy {archives}')
        source_zip = archives[0]
else:
    if not source_archive(source_zip):
        raise RuntimeError(f'SOURCE_ZIP không có cấu trúc repo hợp lệ: {source_zip}')

if repo.exists():
    shutil.rmtree(repo)
if source_dir:
    shutil.copytree(source_dir, repo)
else:
    with zipfile.ZipFile(source_zip) as archive:
        prefix = next(name.removesuffix('pyproject.toml') for name in archive.namelist()
                      if name.endswith('/pyproject.toml'))
        repo.mkdir(parents=True, exist_ok=True)
        for member in archive.infolist():
            if member.is_dir() or not member.filename.startswith(prefix):
                continue
            rel = Path(member.filename[len(prefix):])
            if not rel.parts or rel.is_absolute() or '..' in rel.parts:
                raise RuntimeError(f'Đường dẫn ZIP không hợp lệ: {member.filename}')
            dest = repo / rel
            dest.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(member) as fin, dest.open('wb') as fout:
                shutil.copyfileobj(fin, fout)

assert (repo / 'pyproject.toml').is_file()
expected_sha = hashlib.sha256((repo / 'data/ResearcherBench.jsonl').read_bytes()).hexdigest()
selection = 'original-ids:' + ','.join(QUESTION_IDS)

def valid_manifest(data):
    return (isinstance(data, dict) and data.get('selection') == selection
            and data.get('dataset_sha256') == expected_sha)

def checkpoint_members_ok(names):
    for name in names:
        path = Path(name)
        if path.is_absolute() or '..' in path.parts:
            return False
        if not (name.startswith('output/') or name in ('baseline-runtime.json', 'run_manifest.json')):
            return False
    return True

def checkpoint_zip_ok(path):
    try:
        with zipfile.ZipFile(path) as archive:
            names = archive.namelist()
            if not checkpoint_members_ok(names):
                return False
            manifest = ('run_manifest.json' if 'run_manifest.json' in names
                        else 'output/run_manifest.json')
            return manifest in names and valid_manifest(json.loads(archive.read(manifest)))
    except (OSError, ValueError, zipfile.BadZipFile):
        return False

if not RESUME_ZIP:
    checkpoints = []
    for path in input_dir.rglob('*.zip'):
        if source_zip is not None and path.resolve() == source_zip.resolve():
            continue
        if checkpoint_zip_ok(path):
            checkpoints.append(path)
    for manifest_path in input_dir.rglob('run_manifest.json'):
        try:
            manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
            if not valid_manifest(manifest):
                continue
            folder = manifest_path.parent
            if folder.name == 'output' and (folder / 'run_state').exists():
                checkpoints.append(folder.parent)
            elif (folder / 'output' / 'run_state').exists():
                checkpoints.append(folder)
        except (OSError, ValueError):
            continue
    checkpoints = sorted(set(checkpoints))
    if len(checkpoints) > 1:
        raise RuntimeError(f'Nhiều checkpoint hợp lệ: {checkpoints}; hãy giữ một bản')
    if checkpoints:
        RESUME_ZIP = str(checkpoints[0])

if RESUME_ZIP:
    path = Path(RESUME_ZIP)
    if path.is_dir():
        manifest_path = (path / 'run_manifest.json' if (path / 'run_manifest.json').exists()
                         else path / 'output/run_manifest.json')
        if not valid_manifest(json.loads(manifest_path.read_text(encoding='utf-8'))):
            raise RuntimeError('Checkpoint không khớp QUESTION_IDS hoặc bộ dữ liệu')
        shutil.copytree(path / 'output', Path(OUTPUT_ROOT), dirs_exist_ok=True)
        for name in ('baseline-runtime.json', 'run_manifest.json'):
            if (path / name).is_file():
                shutil.copy2(path / name, Path('/kaggle/working') / name)
    else:
        if not checkpoint_zip_ok(path):
            raise RuntimeError('Checkpoint ZIP không hợp lệ hoặc không khớp QUESTION_IDS')
        with zipfile.ZipFile(path) as archive:
            archive.extractall('/kaggle/working')
print('[SOURCE]', source_dir or source_zip)
print('[CHECKPOINT RESTORED]', RESUME_ZIP or 'None')


In [ ]:
import ast, importlib
source_path = str((repo / "src").resolve())
if source_path in sys.path:
    sys.path.remove(source_path)
sys.path.insert(0, source_path)
importlib.invalidate_caches()
import mrdre_baseline
assert str(Path(mrdre_baseline.__file__).resolve()).startswith(source_path + os.sep)
records = [json.loads(line) for line in (repo / 'data/ResearcherBench.jsonl').read_text(encoding='utf-8').splitlines() if line.strip()]
record_ids = [str(x['id']) for x in records]
assert len(records) == 65, f'Số câu ResearcherBench không đúng: {len(records)}'
assert all(record_ids.count(qid) == 1 for qid in QUESTION_IDS), f'ID không tồn tại hoặc bị trùng: {QUESTION_IDS}'
assert 'mcp==1.29.0' in (repo / 'pyproject.toml').read_text(encoding='utf-8')
assert 'langchain-mcp-adapters==0.3.2' in (repo / 'src/mrdre_baseline/runtime/bootstrap.py').read_text(encoding='utf-8')
assert '--question-ids' in (repo / 'src/mrdre_baseline/runtime/runner.py').read_text(encoding='utf-8')
for p in repo.rglob('*.py'):
    ast.parse(p.read_text(encoding='utf-8'), filename=str(p))
print('[READY] IDs', QUESTION_IDS, '19 stages per question; kernel source:', mrdre_baseline.__file__)
if not shutil.which('zstd'):
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'zstd'], check=True)


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(repo)], check=True)
from mrdre_baseline.services import web as _source_web
assert str(Path(_source_web.__file__).resolve()).startswith(str((repo / "src").resolve()) + os.sep)
print('[READY] MR DRE local package installed; notebook imports verified')


In [ ]:
runtime_env = os.environ.copy()
runtime_env['PYTHONPATH'] = os.pathsep.join(x for x in (str((repo / 'src').resolve()), runtime_env.get('PYTHONPATH', '')) if x)
try:
    subprocess.run([sys.executable, str(repo / 'scripts/bootstrap.py'),
                    '--profile', 'kaggle', '--workdir', '/kaggle/working',
                    '--model', MODEL, '--odr-transport', 'direct', '--odr-ref', ODR_REF],
                   cwd=repo, env=runtime_env, check=True)
except Exception:
    print('=== LAST SERVICE LOGS ===')
    logs = Path('/kaggle/working/baseline-logs')
    for name in ('mcp', 'ollama_gpu0', 'ollama_gpu1', 'langgraph'):
        path = logs / (name + '.log')
        if path.exists():
            print(f'--- {name} ---')
            print('\n'.join(path.read_text(encoding='utf-8', errors='replace').splitlines()[-60:]))
    raise


In [ ]:
# Không bắt đầu nghiên cứu nếu dịch vụ tìm kiếm không trả về nguồn.
import time
# Đường dẫn nguồn đã được cố định ở ô kiểm tra đầu tiên.
from mrdre_baseline.services.web import search_web
error = None
for attempt in range(3):
    try:
        matches = search_web('DeepSeek R1 reinforcement learning reasoning model 2025', 3)
        if not matches:
            raise RuntimeError('DDGS returned zero search results')
        print('[READY] Real web search:', matches[0]['url'])
        error = None
        break
    except Exception as exc:
        error = exc
        print(f'[WEB SEARCH] attempt {attempt + 1}/3: {type(exc).__name__}: {exc}')
        if attempt < 2:
            time.sleep(3 * (attempt + 1))
if error is not None:
    raise RuntimeError('Không có kết quả tìm kiếm web: chưa thể tạo báo cáo có nguồn đáng tin cậy') from error


In [ ]:
command = [sys.executable, '-m', 'mrdre_baseline.runtime.runner',
           '--dataset', 'ResearcherBench', '--question-ids', *QUESTION_IDS,
           '--count', str(COUNT), '--feedback', FEEDBACK, '--extra-eval', EXTRA_EVAL,
           '--retries', str(TASK_RETRIES), '--output-root', OUTPUT_ROOT,
           '--checkpoint-zip', CHECKPOINT_ZIP, '--budget-minutes', str(BUDGET_MINUTES)]
result = subprocess.run(command, cwd=repo, env=runtime_env, check=False)
print('[RUNNER EXIT]', result.returncode)
subprocess.run([sys.executable, '-m', 'mrdre_baseline.runtime.summary',
                '--output-root', OUTPUT_ROOT, '--feedback', FEEDBACK,
                '--extra-eval', EXTRA_EVAL], cwd=repo, env=runtime_env, check=True)
statuses = {}
for qid in QUESTION_IDS:
    state = Path(OUTPUT_ROOT) / 'run_state' / f'{qid}.json'
    statuses[qid] = json.loads(state.read_text()) if state.exists() else {'status': 'missing'}
    print(f'[Q{qid} STATE]', statuses[qid]['status'],
          'stages=', len(statuses[qid].get('completed_tasks', [])))
print('[CHECKPOINT]', CHECKPOINT_ZIP, 'exists=', Path(CHECKPOINT_ZIP).is_file())
if result.returncode != 0:
    raise RuntimeError(f'Chưa hoàn thành đủ {QUESTION_IDS} (exit={result.returncode}). '
                       f'Tải {Path(CHECKPOINT_ZIP).name} trong Output để khôi phục.')
for qid, status in statuses.items():
    if status.get('status') != 'completed' or len(status.get('completed_tasks', [])) != 19:
        raise RuntimeError(f'Câu {qid} chưa hoàn thành đủ 19 bước')
print('[COMPLETE] Original ResearcherBench IDs', QUESTION_IDS, '; all', 19 * COUNT, 'stages finished')
